In [2]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

def call_hf(model_id: str, prompt: str):
    api_url = f"https://api-inference.huggingface.co/models/{model_id}"
    token = os.getenv("HUGGINGFACE_API_KEY")

    assert token and token.startswith("hf_"), "HUGGINGFACE_API_KEY 환경변수가 없거나 형식이 올바르지 않습니다."

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }

    try:
        resp = requests.post(
            api_url,
            headers=headers,
            json={"inputs": prompt, "options": {"wait_for_model": True}},
            timeout=30,
        )
    except requests.RequestException as e:
        print(f"[NETWORK ERROR] {e}")
        return

    print(f"[STATUS] {resp.status_code}")
    ctype = resp.headers.get("content-type", "")
    print(f"[CONTENT-TYPE] {ctype}")

    # JSON이면 파싱, 아니면 원문 일부 출력
    if "application/json" in ctype.lower():
        try:
            data = resp.json()
            print("[JSON]", data)
        except Exception as e:
            print("[JSON PARSE ERROR]", e)
            print("[RAW TEXT PREVIEW]", resp.text[:500])
    else:
        print("[NON-JSON RESPONSE PREVIEW]")
        print(resp.text[:500])

# 1) 문제 보고하신 DialoGPT-medium
print("=== Try DialoGPT-medium ===")
call_hf("microsoft/DialoGPT-medium", "Hello! How are you?")

# 2) 비교용: 보통 잘 응답하는 gpt2 (텍스트 생성)로 재시도
print("\n=== Try gpt2 ===")
call_hf("gpt2", "Hello! How are you?")


=== Try DialoGPT-medium ===
[STATUS] 404
[CONTENT-TYPE] text/plain; charset=utf-8
[NON-JSON RESPONSE PREVIEW]
Not Found

=== Try gpt2 ===
[STATUS] 404
[CONTENT-TYPE] text/plain; charset=utf-8
[NON-JSON RESPONSE PREVIEW]
Not Found
